# HidroXAI-MX — Reporte de cobertura y validación del dataset (OE1)

Genera las **tablas y figuras de validación** que sustentan el *data paper*. Son análisis
**descriptivos y de control de calidad**; no incluyen modelado (eso es Paper 2 y 3).

Si aún no existen los Parquet de `data/processed/`, el notebook **simula** un dataset de
demostración para que todo corra de extremo a extremo (verás un aviso).


In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from hidroxai_mx import report
PROC = ROOT / 'data' / 'processed'
OUT = PROC / 'reportes'; OUT.mkdir(parents=True, exist_ok=True)
print('Salidas ->', OUT)


## 0. Cargar datos (o simular)


In [ ]:
def simular():
    rng = np.random.default_rng(1729)
    fechas = pd.date_range('2010-01-01', '2025-12-31')
    regs = {'12':6, '18':8, '26':6}
    filas = []
    for rh, n in regs.items():
        for j in range(n):
            clave = f'{rh}E{j:03d}'
            lat = 19 + rng.normal(0, 1.5); lon = -101 + rng.normal(0, 1.5)
            estacional = 5 + 4*np.sin(2*np.pi*(fechas.dayofyear/365.0))
            precip = np.clip(estacional + rng.normal(0, 2, len(fechas)), 0, None)
            gasto = np.clip(0.6*pd.Series(precip).shift(3).fillna(0).values + rng.normal(0, 1, len(fechas)), 0, None)
            cal = np.zeros(len(fechas), dtype=int)
            miss = rng.random(len(fechas)) < 0.12; gasto[miss] = np.nan
            cal[(rng.random(len(fechas)) < 0.02)] = 1
            cal[(gasto > np.nanpercentile(gasto, 99.9))] = 2
            filas.append(pd.DataFrame({'clave_estacion':clave,'region_hidrologica':rh,
                'fecha':fechas,'gasto_medio_m3s':gasto,'precip_mm':precip,'calidad':cal,'fuente':'SIH'}))
    return pd.concat(filas, ignore_index=True)

f_hid = PROC / 'series_hidrometricas.parquet'
if f_hid.exists():
    hid = pd.read_parquet(f_hid); DEMO = False
else:
    hid = simular(); DEMO = True
    print('AVISO: usando datos SIMULADOS (no se hallaron Parquet en data/processed).')
hid['fecha'] = pd.to_datetime(hid['fecha'])
print('Filas:', len(hid), '| estaciones:', hid['clave_estacion'].nunique(), '| DEMO =', DEMO)


## 1. Inventario y cobertura
Nº de estaciones por región hidrológica y % de días con dato por estación (periodo 2010–2025).


In [ ]:
inv = report.station_inventory(hid, by='region_hidrologica')
display(inv)
cov = report.coverage_table(hid, 'gasto_medio_m3s', inicio='2010-01-01', fin='2025-12-31')
cov.to_csv(OUT / 'cobertura_por_estacion.csv', index=False)
print('Cobertura media: %.1f%% | estaciones >=80%%: %d/%d' % (
    cov['cobertura'].mean()*100, (cov['cobertura']>=0.8).sum(), len(cov)))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(inv['region_hidrologica'].astype(str), inv['n_estaciones'], color='#7A1737')
ax[0].set_title('Estaciones por región hidrológica'); ax[0].set_xlabel('R.H.')
ax[1].hist(cov['cobertura']*100, bins=20, color='#1F3D5C'); ax[1].axvline(80, color='red', ls='--')
ax[1].set_title('Distribución de cobertura (%)'); ax[1].set_xlabel('% días con dato')
plt.tight_layout(); plt.savefig(OUT / 'fig1_inventario_cobertura.png', dpi=120); plt.show()


## 2. Estadística descriptiva
Rangos, medias y percentiles del objetivo por región y temporada (seca/lluviosa).


In [ ]:
display(hid['gasto_medio_m3s'].describe(percentiles=[.05,.5,.95]).to_frame().T)
d = hid.dropna(subset=['gasto_medio_m3s']).copy()
d['temporada'] = np.where(d['fecha'].dt.month.isin([5,6,7,8,9,10]), 'lluviosa', 'seca')
fig, ax = plt.subplots(figsize=(8,3.6))
grupos = [g['gasto_medio_m3s'].values for _, g in d.groupby('region_hidrologica')]
ax.boxplot(grupos, labels=sorted(d['region_hidrologica'].unique()), showfliers=False)
ax.set_title('Gasto por región hidrológica'); ax.set_xlabel('R.H.'); ax.set_ylabel('m³/s')
plt.tight_layout(); plt.savefig(OUT / 'fig2_descriptiva.png', dpi=120); plt.show()
display(d.groupby('temporada')['gasto_medio_m3s'].describe()[['mean','50%','max']])


## 3. Control de calidad
Conteo de banderas (0=ok, 1=imputado, 2=outlier) y validación del esquema canónico (pandera).


In [ ]:
q = report.quality_summary(hid); display(q)
fig, ax = plt.subplots(figsize=(4.5,3.2))
ax.bar(q['etiqueta'], q['pct'], color=['#2E7D5B','#C9971B','#B23A48'])
ax.set_ylabel('%'); ax.set_title('Banderas de calidad')
plt.tight_layout(); plt.savefig(OUT / 'fig3_calidad.png', dpi=120); plt.show()
try:
    from hidroxai_mx.data import schema
    schema.validate_series(hid.assign(fuente=hid.get('fuente','SIH')))
    print('Validación pandera del esquema canónico: OK')
except Exception as e:
    print('Avisos de validación:', str(e)[:300])


## 4. Consistencia entre fuentes
Correlación y sesgo en estaciones comunes entre dos fuentes (p. ej. SIH vs BANDAS, o SIH vs CLICOM).
Si solo hay una fuente, se omite con un aviso.


In [ ]:
fuentes = hid['fuente'].unique() if 'fuente' in hid else []
if len(fuentes) >= 2:
    a = hid[hid['fuente']==fuentes[0]]; b = hid[hid['fuente']==fuentes[1]]
    agr = report.cross_source_agreement(a, b, 'gasto_medio_m3s')
    display(agr.describe()[['corr','sesgo']] if not agr.empty else agr)
else:
    print('Solo una fuente disponible (', list(fuentes), '). Cuando se integre BANDAS/CLICOM,',
          'esta celda compara correlación y sesgo en estaciones comunes.')


## 5. Coherencia espacial
Ubicación de estaciones (lat/lon) coloreadas por región; densidad por región hidrológica.


In [ ]:
if {'latitud','longitud'}.issubset(hid.columns):
    pts = hid.drop_duplicates('clave_estacion')
    fig, ax = plt.subplots(figsize=(5.5,5))
    for rh, g in pts.groupby('region_hidrologica'):
        ax.scatter(g['longitud'], g['latitud'], s=18, label=f'RH {rh}')
    ax.set_xlabel('lon'); ax.set_ylabel('lat'); ax.legend(); ax.set_title('Estaciones por región')
    plt.tight_layout(); plt.savefig(OUT / 'fig5_mapa.png', dpi=120); plt.show()
else:
    print('Sin lat/lon en el Parquet (únelas desde el catálogo en 04/05 para el mapa).')


## 6. Patrones temporales esperados
Climatología mensual y medias anuales; los años secos deben verse como mínimos (verificación de plausibilidad, no pronóstico).


In [ ]:
mc = report.monthly_climatology(hid, 'gasto_medio_m3s')
an = report.annual_means(hid, 'gasto_medio_m3s')
fig, ax = plt.subplots(1, 2, figsize=(11,3.4))
ax[0].plot(mc['mes'], mc['mean'], marker='o', color='#1F3D5C'); ax[0].set_title('Climatología mensual'); ax[0].set_xlabel('mes')
ax[1].plot(an['anio'], an['gasto_medio_m3s'], marker='o', color='#7A1737'); ax[1].set_title('Media anual'); ax[1].set_xlabel('año')
plt.tight_layout(); plt.savefig(OUT / 'fig6_temporal.png', dpi=120); plt.show()
print('Tip: contrastar los mínimos anuales con sequías históricas conocidas (p.ej. 2011, 2021).')


## 7. Relación inter-variable (precipitación → gasto)
Correlación con rezago: el gasto debe responder a la precipitación con unos días de retraso (coherencia físico-temporal).


In [ ]:
if 'precip_mm' in hid.columns:
    s = report.lagged_corr(hid, 'precip_mm', 'gasto_medio_m3s', max_lag=15)
    fig, ax = plt.subplots(figsize=(7,3.2))
    ax.plot(s.index, s.values, marker='o', color='#2E7D5B')
    ax.set_xlabel('rezago (días)'); ax.set_ylabel('correlación'); ax.set_title('precip(t-lag) vs gasto(t)')
    plt.tight_layout(); plt.savefig(OUT / 'fig7_precip_gasto.png', dpi=120); plt.show()
    print('Rezago de máxima correlación:', int(s.idxmax()), 'días')
else:
    print('precip_mm no presente: usa data/features (precip_idw_mm de 07) para este análisis.')


## 8. Reproducibilidad y procedencia
Hashes de descarga (`_manifest.json`) y resumen del snapshot.


In [ ]:
man = ROOT / 'data' / 'raw' / '_manifest.json'
if man.exists():
    m = json.loads(man.read_text())
    print('Archivos con procedencia registrada:', len(m))
    for k in list(m)[:5]: print(' -', k, m[k]['sha256'][:12], m[k].get('downloaded_at',''))
else:
    print('Sin _manifest.json (se genera al descargar con io.download.fetch).')
print('\nResumen dataset:')
print(' estaciones :', hid['clave_estacion'].nunique())
print(' periodo    :', hid['fecha'].min().date(), '->', hid['fecha'].max().date())
print(' filas      :', len(hid))


## Checklist de figuras/tablas para el *data paper*
- Fig.1 inventario + cobertura · `cobertura_por_estacion.csv`
- Fig.2 estadística descriptiva · Fig.3 calidad
- Fig.5 mapa de estaciones · Fig.6 climatología/medias anuales · Fig.7 precip→gasto
- §4 consistencia entre fuentes (al integrar BANDAS/CLICOM) · §8 procedencia/reproducibilidad

Todas las salidas quedan en `data/processed/reportes/`.
